In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"]=""

In [ ]:
import sys
!{sys.executable} -m pip install -q youtube-transcript-api langchain-community langchain_huggingface faiss-cpu tiktoken python-dotenv yt-dlp


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import subprocess
import re

Step-1a-Indexing (Document Ingestion)

In [100]:

video_id = "Gfr50f6ZBvo"
url = f"https://www.youtube.com/watch?v={video_id}"

# This saves English auto-generated subs into "output.en.srt"
subprocess.run([
    "yt-dlp", "--skip-download",
    "--write-auto-subs", "--sub-lang", "en",
    "--convert-subs", "srt",
    "-o", "output", url
])


CompletedProcess(args=['yt-dlp', '--skip-download', '--write-auto-subs', '--sub-lang', 'en', '--convert-subs', 'srt', '-o', 'output', 'https://www.youtube.com/watch?v=Gfr50f6ZBvo'], returncode=1)

In [111]:
def clean_vtt(vtt_file):
    with open(vtt_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove WEBVTT header & metadata
    text = re.sub(r"WEBVTT.*\n", "", text)
    text = re.sub(r"Kind:.*\n", "", text)
    text = re.sub(r"Language:.*\n", "", text)

    # Remove timestamps like 00:00:01.200 --> 00:00:04.500
    text = re.sub(r"\d{2}:\d{2}:\d{2}\.\d{3} --> .*", "", text)

    # Remove inline word-level timestamps <00:00:00.160>
    text = re.sub(r"<\d{2}:\d{2}:\d{2}\.\d{3}>", "", text)

    # Remove <c> and </c> tags
    text = re.sub(r"</?c>", "", text)

    # Remove extra spaces & newlines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return " ".join(lines)
     

transcript_clean = clean_vtt("output.en.vtt")
print(transcript_clean)  # preview first 1000 chars

the following is a conversation with the following is a conversation with the following is a conversation with demus hasabis demus hasabis demus hasabis ceo and co-founder of deepmind ceo and co-founder of deepmind ceo and co-founder of deepmind a company that has published and builds a company that has published and builds a company that has published and builds some of the most incredible artificial some of the most incredible artificial some of the most incredible artificial intelligence systems in the history of intelligence systems in the history of intelligence systems in the history of computing including alfred zero that computing including alfred zero that computing including alfred zero that learned learned learned all by itself to play the game of gold all by itself to play the game of gold all by itself to play the game of gold better than any human in the world and better than any human in the world and better than any human in the world and alpha fold two that solved prot

Step 1b-Indexing (Text Splitting)

In [119]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([transcript_clean])

In [120]:
len(chunks)

502

In [122]:
chunks[0]

Document(metadata={}, page_content='the following is a conversation with the following is a conversation with the following is a conversation with demus hasabis demus hasabis demus hasabis ceo and co-founder of deepmind ceo and co-founder of deepmind ceo and co-founder of deepmind a company that has published and builds a company that has published and builds a company that has published and builds some of the most incredible artificial some of the most incredible artificial some of the most incredible artificial intelligence systems in the history of intelligence systems in the history of intelligence systems in the history of computing including alfred zero that computing including alfred zero that computing including alfred zero that learned learned learned all by itself to play the game of gold all by itself to play the game of gold all by itself to play the game of gold better than any human in the world and better than any human in the world and better than any human in the world

Step 1 & 1d - Indexing(Embedding Generation and Storing in Vector Store)

In [123]:
embedding= HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(chunks,embedding)

In [124]:
vector_store.index_to_docstore_id

{0: '8bad2266-9616-490a-ade1-a51114068cef',
 1: '5b6c155e-6e26-4b6a-9036-627cec8fab6b',
 2: 'e1c0e75e-9873-4c1b-be25-d7def0faca83',
 3: 'ce55fc76-ce85-4073-934d-3c11c1a1219b',
 4: '5ebde308-280d-4ce8-9e42-3b398d4dc129',
 5: '0848d38d-e793-4320-aa56-51bc8c6b32e3',
 6: '06644d28-7677-4758-9904-d93332557804',
 7: '0093ac1d-4808-49a7-9e50-ddb75a53566c',
 8: '217e843c-c953-4c49-8062-dd58b1cab642',
 9: '31021f71-6369-45ee-8c8d-0337017fc5ad',
 10: '387bbddd-61ee-4ec6-b11a-a2edaf9bc6fe',
 11: '47c29902-2471-4c93-a4f0-01e55a427c66',
 12: '33e8acbd-1eb4-4df1-82a9-27f97cc4e63e',
 13: 'f6ed041b-117d-44e9-b1ff-e6a17562c17a',
 14: '61689ab0-dc73-4c37-8e91-b66538fa9096',
 15: '7ff5725f-ef55-4ae6-9233-4584b8bef7dc',
 16: 'ad25dca8-92ef-4644-b748-7f19f8e3ccab',
 17: '79e44660-cf8b-4175-ac75-36174eccc438',
 18: 'a3cf5ccf-93f5-4efc-bf86-3160cc8f860b',
 19: '2ce93e45-fbca-480f-8d80-d54e32a5d37e',
 20: '0bf3edf2-442f-4c7f-afaf-bf024a409852',
 21: '308b67aa-d207-45a4-b143-f8670ae9a9c3',
 22: 'f26c1661-ead1-

In [126]:
vector_store.get_by_ids(['7e21c02a-3f09-4710-8c29-97a123972346'])

[Document(id='7e21c02a-3f09-4710-8c29-97a123972346', metadata={}, page_content='our sponsors in the description and now let me leave you with some words and now let me leave you with some words and now let me leave you with some words from edskar dykstra from edskar dykstra from edskar dykstra computer science is no more about computer science is no more about computer science is no more about computers computers computers than astronomy is about telescopes than astronomy is about telescopes than astronomy is about telescopes thank you for listening and hope to see thank you for listening and hope to see thank you for listening and hope to see you next time')]

Step-2 Retrieval

In [127]:
retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [128]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002C95CE5C150>, search_kwargs={'k': 4})

In [129]:
retriever.invoke("what is deepmind")

[Document(id='3419ff96-0807-4447-91fd-763167150f32', metadata={}, page_content="deep mind from we've used of ai is in deep mind from we've used of ai is in deep mind from the beginning which is using games as a the beginning which is using games as a the beginning which is using games as a testing ground for proving out ai testing ground for proving out ai testing ground for proving out ai algorithms and developing ai algorithms algorithms and developing ai algorithms algorithms and developing ai algorithms and that was a that was a sort of um a and that was a that was a sort of um a and that was a that was a sort of um a core component of our vision at the core component of our vision at the core component of our vision at the start of deepmind was that we would use start of deepmind was that we would use start of deepmind was that we would use games very heavily uh as our main games very heavily uh as our main games very heavily uh as our main testing ground certainly to begin with t

Step-3 Augmentation

In [130]:
prompt=PromptTemplate(
    template="""
    You are a helpful assistance.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}
    Question: {question}
""",
input_variables=['context','question']
)

In [132]:
question="Is the topic of aliens discussed in the video? If yes then what was discussed?"
retrieved_docs=retriever.invoke(question)

In [135]:
retrieved_docs

[Document(id='0366b031-23c1-4135-b641-379580fd21be', metadata={}, page_content="of voices and have joined that cacophony of voices and what we did we opened our ears and we what we did we opened our ears and we what we did we opened our ears and we heard nothing heard nothing heard nothing and many people who argue that there are and many people who argue that there are and many people who argue that there are aliens would say well we haven't really aliens would say well we haven't really aliens would say well we haven't really done exhaustive search yet and maybe done exhaustive search yet and maybe done exhaustive search yet and maybe we're looking in the wrong bands and and we're looking in the wrong bands and and we're looking in the wrong bands and and we've got the wrong devices and we we've got the wrong devices and we we've got the wrong devices and we wouldn't notice what an alien form was wouldn't notice what an alien form was wouldn't notice what an alien form was like to be

In [136]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"of voices and have joined that cacophony of voices and what we did we opened our ears and we what we did we opened our ears and we what we did we opened our ears and we heard nothing heard nothing heard nothing and many people who argue that there are and many people who argue that there are and many people who argue that there are aliens would say well we haven't really aliens would say well we haven't really aliens would say well we haven't really done exhaustive search yet and maybe done exhaustive search yet and maybe done exhaustive search yet and maybe we're looking in the wrong bands and and we're looking in the wrong bands and and we're looking in the wrong bands and and we've got the wrong devices and we we've got the wrong devices and we we've got the wrong devices and we wouldn't notice what an alien form was wouldn't notice what an alien form was wouldn't notice what an alien form was like to be so different to what we're like to be so different to what we're like to be so

In [138]:
final_prompt=prompt.invoke({"context": context_text, "question": question})

In [139]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistance.\n    Answer ONLY from the provided transcript context.\n    If the context is insufficient, just say you don't know.\n\n    of voices and have joined that cacophony of voices and what we did we opened our ears and we what we did we opened our ears and we what we did we opened our ears and we heard nothing heard nothing heard nothing and many people who argue that there are and many people who argue that there are and many people who argue that there are aliens would say well we haven't really aliens would say well we haven't really aliens would say well we haven't really done exhaustive search yet and maybe done exhaustive search yet and maybe done exhaustive search yet and maybe we're looking in the wrong bands and and we're looking in the wrong bands and and we're looking in the wrong bands and and we've got the wrong devices and we we've got the wrong devices and we we've got the wrong devices and we wouldn't notice what an

Step-4 - Generation

In [140]:
llm= HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm)

In [142]:
answer=model.invoke(final_prompt)
print(answer.content)

Yes, the topic of aliens is discussed.  The speaker discusses how the idea of aliens is sometimes used as an argument against the development of space travel and potentially uniting Earth's political and economic complex, as the present lack of empirical evidence for extraterrestrial life makes denying the possibility of a multi-planetary species moot. The debate revolves around the need for a complete examination of all existing possibilities before taking long leaps of speculation.  The speaker also brings up hope for AI, which could contribute significantly to humanity's development and likely lead to a proliferated scientific understanding that might, finally, reveal something truly unexpected.  Additionally, the speaker, though not entirely convincingly, suggests a proposition that consciousness itself might originate from somewhere other than Earth. 


Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [144]:
def format_docs(Retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [145]:
parallel_chain=RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [146]:
parallel_chain.invoke('who is Demis')

{'context': "of voices and have joined that cacophony of voices and what we did we opened our ears and we what we did we opened our ears and we what we did we opened our ears and we heard nothing heard nothing heard nothing and many people who argue that there are and many people who argue that there are and many people who argue that there are aliens would say well we haven't really aliens would say well we haven't really aliens would say well we haven't really done exhaustive search yet and maybe done exhaustive search yet and maybe done exhaustive search yet and maybe we're looking in the wrong bands and and we're looking in the wrong bands and and we're looking in the wrong bands and and we've got the wrong devices and we we've got the wrong devices and we we've got the wrong devices and we wouldn't notice what an alien form was wouldn't notice what an alien form was wouldn't notice what an alien form was like to be so different to what we're like to be so different to what we're l

In [148]:
parser=StrOutputParser()

In [149]:
main_chain=parallel_chain | prompt | model | parser

In [150]:
main_chain.invoke("Can you summerize the video")

"The video addresses the question of whether there are aliens and the implications of this discovery. The speaker emphasizes the idea that observing and understanding extraterrestrial life is key to surviving as a species. They point out the challenges and limitations human civilization faces in achieving this goal.   The speaker also highlights the boundless potential of AI and technology by presenting it as the greatest benefit to humankind and envisions how it could solve many of humanity's problems.  Finally, the speaker touches upon the possibility that consciousness itself originates from other life forms and reflects on equal level of development and alien species. \n"